In [1]:
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Dict, Any
import pandas as pd
import numpy as np
import math
from typing import Callable, List, Optional

In [5]:
def df_to_bin_tensors(df: pd.DataFrame, max_particles: int = 200) -> Tuple[torch.LongTensor, torch.LongTensor, torch.LongTensor, torch.LongTensor]:
    """
    Liefert pT, eta, phi Tensors (B, N) mit -1 als Padding, auf CPU (nicht auf device).
    Effizientere Implementation: nur vorhandene Spalten werden genutzt.
    """
    pattern = re.compile(r'^(pT|eta|phi)_bin_(\d+)$')
    present_idxs = set()
    for col in df.columns:
        m = pattern.match(col)
        if m:
            present_idxs.add(int(m.group(2)))
    max_present = max(present_idxs) if present_idxs else -1

    B = len(df)
    N = max_particles

    pT_arr = np.full((B, N), -1, dtype=np.int64)
    eta_arr = np.full((B, N), -1, dtype=np.int64)
    phi_arr = np.full((B, N), -1, dtype=np.int64)

    for j in sorted(present_idxs):
        if j >= N:
            break
        pt_col = f"pT_bin_{j}"
        if pt_col in df.columns:
            pT_arr[:, j] = df[pt_col].fillna(-1).astype(np.int64).values
        eta_col = f"eta_bin_{j}"
        if eta_col in df.columns:
            eta_arr[:, j] = df[eta_col].fillna(-1).astype(np.int64).values
        phi_col = f"phi_bin_{j}"
        if phi_col in df.columns:
            phi_arr[:, j] = df[phi_col].fillna(-1).astype(np.int64).values

    if max_present >= 0:
        available = [f"pT_bin_{j}" for j in range(min(max_present + 1, N)) if f"pT_bin_{j}" in df.columns]
        if available:
            stacked = np.stack([df[c].fillna(-1).astype(np.int64).values for c in available], axis=1)  # (B, M)
            counts = np.sum(stacked != -1, axis=1).astype(np.int64)
        else:
            counts = np.zeros(B, dtype=np.int64)
    else:
        counts = np.zeros(B, dtype=np.int64)

    counts = np.minimum(counts, N)

    pT_t = torch.from_numpy(pT_arr).long()   # CPU tensor
    eta_t = torch.from_numpy(eta_arr).long()
    phi_t = torch.from_numpy(phi_arr).long()
    counts_t = torch.from_numpy(counts).long()

    return pT_t, eta_t, phi_t, counts_t

class ParticleEmbedder(nn.Module):
    def __init__(self,
                 emb_dim: int = 256,
                 pT_max_value: int = 40,
                 eta_max_value: int = 30,
                 phi_max_value: int = 30,
                 max_particles: int = 100,
                 use_position_embedding: bool = False,
                 dropout: float = 0.0):
        super().__init__()
        self.emb_dim = emb_dim
        self.max_particles = max_particles
        self.S = 1 + max_particles + 1  # START + N + STOP

        self.pT_emb = nn.Embedding(num_embeddings=pT_max_value + 2, embedding_dim=emb_dim, padding_idx=0)
        self.eta_emb = nn.Embedding(num_embeddings=eta_max_value + 2, embedding_dim=emb_dim, padding_idx=0)
        self.phi_emb = nn.Embedding(num_embeddings=phi_max_value + 2, embedding_dim=emb_dim, padding_idx=0)

        self.use_position_embedding = use_position_embedding
        if use_position_embedding:
            self.pos_emb = nn.Embedding(max_particles + 1, emb_dim)
            pos_idx = torch.arange(1, max_particles + 1).unsqueeze(0)  # (1, N)
            self.register_buffer("pos_idx_buffer", pos_idx, persistent=False)
        else:
            self.pos_emb = None
            self.register_buffer("pos_idx_buffer", torch.empty(0), persistent=False)

        self.start_token = nn.Parameter(torch.randn(emb_dim) * 0.02)
        self.stop_token = nn.Parameter(torch.randn(emb_dim) * 0.02)

        self.layernorm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, pT_bins: torch.LongTensor, eta_bins: torch.LongTensor, phi_bins: torch.LongTensor, counts: torch.LongTensor) -> torch.FloatTensor:
        B, N = pT_bins.shape
        assert N == self.max_particles
        device = pT_bins.device
        D = self.emb_dim
        S = self.S

        pT_idx = torch.clamp(pT_bins + 1, min=0, max=self.pT_emb.num_embeddings - 1)
        eta_idx = torch.clamp(eta_bins + 1, min=0, max=self.eta_emb.num_embeddings - 1)
        phi_idx = torch.clamp(phi_bins + 1, min=0, max=self.phi_emb.num_embeddings - 1)

        e_pT = self.pT_emb(pT_idx)
        e_eta = self.eta_emb(eta_idx)
        e_phi = self.phi_emb(phi_idx)
        e_particles = e_pT + e_eta + e_phi

        if self.use_position_embedding:
            pos_idx = self.pos_idx_buffer.to(device).expand(B, N)  # (B,N)
            e_particles = e_particles + self.pos_emb(pos_idx)

        seq = torch.zeros((B, S, D), dtype=e_particles.dtype, device=device)
        seq[:, 0, :] = self.start_token.view(1, D)

        arange = torch.arange(N, device=device).unsqueeze(0).expand(B, N)
        counts_exp = counts.unsqueeze(1).expand(B, N)
        shift = (arange >= counts_exp).long()
        dest = 1 + arange + shift  # (B, N)

        dest_idx = dest.unsqueeze(-1).expand(B, N, D)
        seq = seq.scatter(dim=1, index=dest_idx, src=e_particles)

        stop_needed = (counts < N)
        if stop_needed.any():
            stop_pos = (1 + counts).to(torch.long)
            batch_idx = torch.arange(B, device=device)
            seq[batch_idx[stop_needed], stop_pos[stop_needed], :] = self.stop_token.view(1, D)

        seq = self.layernorm(seq)
        seq = self.dropout(seq)
        return seq

    def make_masks(self, pT_bins: torch.LongTensor, counts: torch.LongTensor) -> Tuple[torch.BoolTensor, torch.BoolTensor]:
        B, N = pT_bins.shape
        device = pT_bins.device
        S = self.S

        orig_pad = (pT_bins == -1)
        arange = torch.arange(N, device=device).unsqueeze(0).expand(B, N)
        counts_exp = counts.unsqueeze(1).expand(B, N)
        shift = (arange >= counts_exp).long()
        dest = 1 + arange + shift

        src_key_padding_mask = torch.ones((B, S), dtype=torch.bool, device=device)
        src_key_padding_mask[:, 0] = False  # START unmasked

        batch_idx = torch.arange(B, device=device).unsqueeze(1).expand(B, N)
        real_mask = ~orig_pad
        if real_mask.any():
            src_key_padding_mask[batch_idx[real_mask], dest[real_mask]] = False

        stop_needed = (counts < N)
        if stop_needed.any():
            stop_pos = (1 + counts)
            batch_idx_simple = torch.arange(B, device=device)
            src_key_padding_mask[batch_idx_simple[stop_needed], stop_pos[stop_needed]] = False

        causal_mask = torch.triu(torch.ones((S, S), dtype=torch.bool, device=device), diagonal=0)
        causal_mask[:, 0] = False
        causal_mask[0, 0] = False

        return src_key_padding_mask, causal_mask

class ParticleTransformer(nn.Module):
    def __init__(self,
                 num_pT_embeddings: int = 42,
                 num_eta_embeddings: int = 32,
                 num_phi_embeddings: int = 32,
                 emb_dim: int = 256,
                 n_layers: int = 8,
                 n_heads: int = 4,
                 dim_feedforward: int = None,
                 dropout: float = 0.1,
                 pad_token: int = 0):
        super().__init__()
        self.emb_dim = emb_dim
        self.pad_token = pad_token
        if dim_feedforward is None:
            dim_feedforward = emb_dim * 4

        encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim,
                                                   nhead=n_heads,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout,
                                                   activation='relu',
                                                   batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, norm=nn.LayerNorm(emb_dim))

        self.final_norm = nn.LayerNorm(emb_dim)
        self.final_dropout = nn.Dropout(dropout)

        self.type_head = nn.Linear(emb_dim, 3)
        self.head_pT  = nn.Linear(emb_dim, num_pT_embeddings)
        self.head_eta = nn.Linear(emb_dim, num_eta_embeddings)
        self.head_phi = nn.Linear(emb_dim, num_phi_embeddings)

    def forward(self, seq_emb: torch.FloatTensor, src_key_padding_mask: torch.BoolTensor, causal_mask: torch.BoolTensor):
        B, S, D = seq_emb.shape
        assert D == self.emb_dim

        attn_mask = None
        if causal_mask is not None:
            attn_mask = causal_mask
            if attn_mask.dtype != torch.bool:
                attn_mask = attn_mask.bool()
            assert attn_mask.shape == (S, S)

        enc = self.encoder(seq_emb, mask=attn_mask, src_key_padding_mask=src_key_padding_mask)
        enc = self.final_norm(enc)
        enc = self.final_dropout(enc)

        logits_type = self.type_head(enc)
        logits_pT  = self.head_pT(enc)
        logits_eta = self.head_eta(enc)
        logits_phi = self.head_phi(enc)

        return logits_type, logits_pT, logits_eta, logits_phi

def load_checkpoint_maybe_with_config(path: str, device: torch.device) -> Dict[str, Any]:
    """
    Lädt ein .pt-File und versucht, entweder:
      - direkt ein state_dict zu liefern (-> returns {"state_dict": ...})
      - oder ein checkpoint dict mit keys like 'model_state_dict' und 'config'
    """
    ckpt = torch.load(path, map_location=device)
    # flexible handling
    if isinstance(ckpt, dict):
        # if it looks like a model state dict already
        if 'model_state_dict' in ckpt or 'state_dict' in ckpt or 'config' in ckpt:
            return ckpt
        else:
            # maybe the user saved state_dict directly
            return {'model_state_dict': ckpt}
    else:
        # e.g. saved as state_dict tensor store
        return {'model_state_dict': ckpt}


def build_models_from_config(config: Dict[str, Any]) -> Tuple[ParticleEmbedder, ParticleTransformer]:
    """
    Erzeugt embedder und transformer aus einem config-dict.
    Erwartete keys (falls nicht vorhanden, werden Defaultwerte genutzt):
      - emb_dim, pT_max, eta_max, phi_max, max_particles, use_position_embedding, dropout
      - num_pT_embeddings, num_eta_embeddings, num_phi_embeddings, n_layers, n_heads, dim_feedforward
    """
    emb_conf = {
        'emb_dim': config.get('emb_dim', 256),
        'pT_max_value': config.get('pT_max_value', config.get('num_pT_embeddings', 40)),
        'eta_max_value': config.get('eta_max_value', config.get('num_eta_embeddings', 30)),
        'phi_max_value': config.get('phi_max_value', config.get('num_phi_embeddings', 30)),
        'max_particles': config.get('max_particles', 100),
        'use_position_embedding': config.get('use_position_embedding', False),
        'dropout': config.get('dropout', 0.0)
    }
    embedder = ParticleEmbedder(**emb_conf)

    trans_conf = {
        'num_pT_embeddings': config.get('num_pT_embeddings', 42),
        'num_eta_embeddings': config.get('num_eta_embeddings', 32),
        'num_phi_embeddings': config.get('num_phi_embeddings', 32),
        'emb_dim': config.get('emb_dim', 256),
        'n_layers': config.get('n_layers', 8),
        'n_heads': config.get('n_heads', 4),
        'dim_feedforward': config.get('dim_feedforward', None),
        'dropout': config.get('dropout', 0.1),
        'pad_token': config.get('pad_token', 0)
    }
    transformer = ParticleTransformer(**trans_conf)
    return embedder, transformer


def load_model_from_checkpoint(path: str, device: torch.device, config_override: Dict[str, Any] = None):
    ckpt = torch.load(
        path,
        map_location=device,
        weights_only=True
    )

    # determine config
    config = {}
    if 'config' in ckpt and isinstance(ckpt['config'], dict):
        config.update(ckpt['config'])
    if config_override:
        config.update(config_override)

    embedder, transformer = build_models_from_config(config)
    wrapper = nn.Module()
    # put both as attributes for easy loading
    wrapper.embedder = embedder
    wrapper.transformer = transformer

    # load weights: support both single state_dict or separate keys
    if 'model_state_dict' in ckpt:
        state = ckpt['model_state_dict']
    elif 'state_dict' in ckpt:
        state = ckpt['state_dict']
    else:
        # maybe user provided a raw state_dict
        state = ckpt

    # Try loading by splitting keys if checkpoint stored a flat dict with prefix
    try:
        wrapper_state = wrapper.state_dict()
        # align keys if prefixes exist
        # naive approach: if keys match directly -> load
        missing, unexpected = wrapper.load_state_dict(state, strict=False)
    except Exception as e:
        # fallback: try to load into embedder / transformer separately
        if 'embedder.' in list(state.keys())[0]:
            # state already prefixed
            wrapper.load_state_dict(state, strict=False)
        else:
            # try matching embedder and transformer prefixes
            s_embed = {k.replace('embedder.', ''): v for k, v in state.items() if k.startswith('embedder.')}
            s_trans = {k.replace('transformer.', ''): v for k, v in state.items() if k.startswith('transformer.')}
            if s_embed:
                embedder.load_state_dict(s_embed, strict=False)
            if s_trans:
                transformer.load_state_dict(s_trans, strict=False)
    wrapper.to(device)
    wrapper.eval()
    return wrapper, config



def _topk_sample_from_logits(logits: torch.Tensor, top_k: int = 5000, banned_indices: Optional[List[int]] = None) -> int:
    """
    Sample one index from `logits` (1D tensor) using top-k sampling.
    - logits: 1D tensor of raw logits (not log-probs).
    - top_k: keep top_k logits (if top_k >= vocab_size -> keep all).
    - banned_indices: list of indices to forbid (set logit = -inf).
    Rückgabe: ausgewählter Index (int).
    """
    if banned_indices:
        # mask banned positions
        logits = logits.clone()
        for idx in banned_indices:
            if 0 <= idx < logits.size(0):
                logits[idx] = -1e9

    vocab_size = logits.size(0)
    k = min(max(1, int(top_k)), vocab_size)
    if k < vocab_size:
        vals, idxs = torch.topk(logits, k)
        probs = F.softmax(vals, dim=0)
        choice = torch.multinomial(probs, num_samples=1).item()
        selected = int(idxs[choice].item())
        return selected
    else:
        probs = F.softmax(logits, dim=0)
        selected = int(torch.multinomial(probs, num_samples=1).item())
        return selected

In [6]:
def load_model_from_checkpoint(path: str, device: torch.device, config_override: Dict[str, Any] = None):
    ckpt = torch.load(
        path,
        map_location=device,
        weights_only=True
    )

    # determine config
    config = {}
    if 'config' in ckpt and isinstance(ckpt['config'], dict):
        config.update(ckpt['config'])
    if config_override:
        config.update(config_override)

    embedder, transformer = build_models_from_config(config)
    wrapper = nn.Module()
    # put both as attributes for easy loading
    wrapper.embedder = embedder
    wrapper.transformer = transformer

    # load weights: support both single state_dict or separate keys
    if 'model_state_dict' in ckpt:
        state = ckpt['model_state_dict']
    elif 'state_dict' in ckpt:
        state = ckpt['state_dict']
    else:
        # maybe user provided a raw state_dict
        state = ckpt

    # Try loading by splitting keys if checkpoint stored a flat dict with prefix
    try:
        wrapper_state = wrapper.state_dict()
        # align keys if prefixes exist
        # naive approach: if keys match directly -> load
        missing, unexpected = wrapper.load_state_dict(state, strict=False)
    except Exception as e:
        # fallback: try to load into embedder / transformer separately
        if 'embedder.' in list(state.keys())[0]:
            # state already prefixed
            wrapper.load_state_dict(state, strict=False)
        else:
            # try matching embedder and transformer prefixes
            s_embed = {k.replace('embedder.', ''): v for k, v in state.items() if k.startswith('embedder.')}
            s_trans = {k.replace('transformer.', ''): v for k, v in state.items() if k.startswith('transformer.')}
            if s_embed:
                embedder.load_state_dict(s_embed, strict=False)
            if s_trans:
                transformer.load_state_dict(s_trans, strict=False)
    wrapper.to(device)
    wrapper.eval()
    return wrapper, config

checkpoint_path = "/home/ew640340/Ph.D./Foundation_Models/checkpoints/best_model_weights.pt" 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config_override = {
    'emb_dim': 256,
    'num_pT_embeddings': 42,
    'num_eta_embeddings': 32,
    'num_phi_embeddings': 32,
    'max_particles': 100,
    'n_layers': 8,
    'n_heads': 4,
    'dropout': 0.1,
}

wrapper, config_used = load_model_from_checkpoint(checkpoint_path, device, config_override=config_override)

In [ ]:
def sample_jets_conditional(
    wrapper: torch.nn.Module,
    device: torch.device,
    num_samples: int = 10,
    max_particles: int = 100,
    top_k: int = 5000,
    seed: Optional[int] = None,
    compute_logprob_func: Optional[Callable] = None
) -> Dict[str, torch.Tensor]:
    """
    Sequentielles konditionales Sampling pro Position:
      - type -> falls particle:
          - sample pT, setze pT, recompute
          - sample eta, setze eta, recompute
          - sample phi, setze phi, increment counts
      - stop beendet Sequenz
    """
    if seed is not None:
        torch.manual_seed(seed)

    wrapper.eval()
    embedder = wrapper.embedder
    transformer = wrapper.transformer

    pT_all = torch.full((num_samples, max_particles), fill_value=-1, dtype=torch.long, device=device)
    eta_all = torch.full((num_samples, max_particles), fill_value=-1, dtype=torch.long, device=device)
    phi_all = torch.full((num_samples, max_particles), fill_value=-1, dtype=torch.long, device=device)
    counts_all = torch.zeros((num_samples,), dtype=torch.long, device=device)

    S = 1 + max_particles + 1
    causal_mask = torch.triu(torch.ones((S, S), dtype=torch.bool, device=device), diagonal=1)

    with torch.no_grad():
        for i in range(num_samples):
            pT_bins = torch.full((1, max_particles), -1, dtype=torch.long, device=device)
            eta_bins = torch.full((1, max_particles), -1, dtype=torch.long, device=device)
            phi_bins = torch.full((1, max_particles), -1, dtype=torch.long, device=device)
            counts = torch.tensor([0], dtype=torch.long, device=device)

            for step in range(max_particles):
                # 1) Forward um type vorherzusagen (auf Basis bisheriger Partikel)
                seq_emb = embedder(pT_bins, eta_bins, phi_bins, counts)
                src_key_padding_mask, _ = embedder.make_masks(pT_bins, counts)
                logits_type, logits_pT, logits_eta, logits_phi = transformer(seq_emb, src_key_padding_mask, causal_mask)
                pos = int(1 + int(counts.item()))

                # sample type
                logits_type_pos = logits_type[0, pos, :]
                sampled_type = _topk_sample_from_logits(logits_type_pos, top_k=top_k)

                if sampled_type == 2:
                    # stop
                    break
                elif sampled_type == 0:
                    # ungewöhnlich: start an non-zero pos -> breche ab
                    break
                else:
                    # particle -> nun sequentiell die Attribute sampeln, jeweils conditioned
                    cur_idx = int(counts.item())

                    # --- pT ---
                    logits_pT_pos = logits_pT[0, pos, :].clone()
                    if logits_pT_pos.numel() > 1:
                        logits_pT_pos[0] = -1e9
                    sampled_pT_idx = _topk_sample_from_logits(logits_pT_pos, top_k=top_k)
                    # map back to original bin
                    pT_val = int(sampled_pT_idx) - 1
                    pT_bins[0, cur_idx] = pT_val

                    # recompute nach pT gesetzt
                    seq_emb = embedder(pT_bins, eta_bins, phi_bins, counts)
                    src_key_padding_mask, _ = embedder.make_masks(pT_bins, counts)
                    _, logits_pT2, logits_eta2, logits_phi2 = transformer(seq_emb, src_key_padding_mask, causal_mask)

                    # --- eta (konditioniert auf pT) ---
                    logits_eta_pos = logits_eta2[0, pos, :].clone()
                    if logits_eta_pos.numel() > 1:
                        logits_eta_pos[0] = -1e9
                    sampled_eta_idx = _topk_sample_from_logits(logits_eta_pos, top_k=top_k)
                    eta_val = int(sampled_eta_idx) - 1
                    eta_bins[0, cur_idx] = eta_val

                    # recompute nach eta gesetzt
                    seq_emb = embedder(pT_bins, eta_bins, phi_bins, counts)
                    src_key_padding_mask, _ = embedder.make_masks(pT_bins, counts)
                    _, _, _, logits_phi3 = transformer(seq_emb, src_key_padding_mask, causal_mask)

                    # --- phi (konditioniert auf pT,eta) ---
                    logits_phi_pos = logits_phi3[0, pos, :].clone()
                    if logits_phi_pos.numel() > 1:
                        logits_phi_pos[0] = -1e9
                    sampled_phi_idx = _topk_sample_from_logits(logits_phi_pos, top_k=top_k)
                    phi_val = int(sampled_phi_idx) - 1
                    phi_bins[0, cur_idx] = phi_val

                    # Jetzt zählen wir das Partikel endgültig
                    counts = torch.tensor([cur_idx + 1], dtype=torch.long, device=device)

            # Speichere das Sample
            pT_all[i] = pT_bins[0]
            eta_all[i] = eta_bins[0]
            phi_all[i] = phi_bins[0]
            counts_all[i] = counts.item()

    result = {
        'pT_bins': pT_all.cpu(),
        'eta_bins': eta_all.cpu(),
        'phi_bins': phi_all.cpu(),
        'counts': counts_all.cpu()
    }

    if compute_logprob_func is not None:
        with torch.no_grad():
            lp = compute_logprob_func(wrapper, pT_all.to(device), eta_all.to(device), phi_all.to(device), counts_all.to(device), device)
        for k, v in lp.items():
            result[k] = v.cpu() if torch.is_tensor(v) else v

    return result


In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    wrapper.to(device)
    samples = sample_jets_conditional(wrapper=wrapper, device=device, num_samples=20, max_particles=100, top_k=5000, compute_logprob_func=None, seed=42)
    print("Generated counts:", samples['counts'])
    print("pT sample (first):", samples['pT_bins'][0])

Generated counts: tensor([0, 2, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0])
pT sample (first): tensor([14,  7, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1,
        -1, -1, -1, -1, -1, -1, -1, -1, -1, -1])
